In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import issparse, load_npz
from sklearn.metrics.pairwise import cosine_similarity

# find the data folder whether the notebook runs from the project root or from notebooks/
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"
print("Using data folder:", DATA)

Using data folder: c:\Users\mansi bramta\OneDrive\Desktop\Hybrid_Recommedation_System\hybrid_recommedation_system\data


In [2]:
songs_data = pd.read_csv(DATA / "collab_filtered_data.csv")
track_ids = np.load(DATA / "track_ids.npy", allow_pickle=True)
transformed_matrix = load_npz(DATA / "transformed_hybrid_data.npz")
interaction_matrix = load_npz(DATA / "interaction_matrix.npz")

print("songs_data:", songs_data.shape)
print("track_ids:", track_ids.shape)
print("transformed_matrix:", transformed_matrix.shape)
print("interaction_matrix:", interaction_matrix.shape)

# all three must list the songs in the same order
assert (songs_data["track_id"].to_numpy().astype(str) == track_ids.astype(str)).all()

songs_data: (15974, 19)
track_ids: (15974,)
transformed_matrix: (15974, 3973)
interaction_matrix: (15974, 962037)


In [3]:
class HybridRecommenderSystem:
    """Mixes content-based similarity with collaborative (listening-history) similarity."""

    def __init__(self, number_of_recommendations: int, weight_content_based: float):
        if not 0 <= weight_content_based <= 1:
            raise ValueError("weight_content_based must be between 0 and 1.")
        self.number_of_recommendations = number_of_recommendations
        self.weight_content_based = weight_content_based
        self.weight_collaborative = 1 - weight_content_based

    # ------------------------------------------------------------------ helpers
    @staticmethod
    def _find_song_position(songs_data, song_name, artist_name):
        """Row position of the song in songs_data (case-insensitive)."""
        song_name = song_name.lower().strip()
        artist_name = artist_name.lower().strip()
        mask = (songs_data["name"] == song_name) & (songs_data["artist"] == artist_name)
        positions = np.flatnonzero(mask.to_numpy())
        if len(positions) == 0:
            raise ValueError(f"'{song_name}' by '{artist_name}' not found in the data.")
        return positions[0]

    @staticmethod
    def _check_alignment(songs_data, track_ids, transformed_matrix, interaction_matrix):
        """The three data structures must list the songs in exactly the same order."""
        n = len(songs_data)
        if not (len(track_ids) == transformed_matrix.shape[0] == interaction_matrix.shape[0] == n):
            raise ValueError("songs_data, track_ids and the matrices have different numbers of rows.")
        if not np.array_equal(songs_data["track_id"].to_numpy().astype(str), track_ids.astype(str)):
            raise ValueError("songs_data and track_ids are in a different order. Rebuild the data files.")

    def _calculate_content_based_similarities(self, position, transformed_matrix):
        input_vector = transformed_matrix[position]
        if not issparse(input_vector):
            input_vector = input_vector.reshape(1, -1)
        return cosine_similarity(input_vector, transformed_matrix).ravel()

    def _calculate_collaborative_filtering_similarities(self, position, interaction_matrix):
        return cosine_similarity(interaction_matrix[position], interaction_matrix).ravel()

    @staticmethod
    def _normalize_similarities(similarity_scores):
        minimum = np.min(similarity_scores)
        maximum = np.max(similarity_scores)
        if maximum == minimum:  # avoid dividing by zero
            return np.zeros_like(similarity_scores)
        return (similarity_scores - minimum) / (maximum - minimum)

    def _weighted_combination(self, content_based_scores, collaborative_filtering_scores):
        return (self.weight_content_based * content_based_scores
                + self.weight_collaborative * collaborative_filtering_scores)

    # --------------------------------------------------------------------- main
    def give_recommendations(self, song_name, artist_name, songs_data, track_ids,
                             transformed_matrix, interaction_matrix):
        self._check_alignment(songs_data, track_ids, transformed_matrix, interaction_matrix)
        position = self._find_song_position(songs_data, song_name, artist_name)

        content = self._calculate_content_based_similarities(position, transformed_matrix)
        collaborative = self._calculate_collaborative_filtering_similarities(position, interaction_matrix)

        scores = self._weighted_combination(
            self._normalize_similarities(content),
            self._normalize_similarities(collaborative),
        )
        scores[position] = np.inf  # the chosen song always comes first

        top = np.argsort(scores)[::-1][: self.number_of_recommendations + 1]
        return songs_data.iloc[top].drop(columns=["track_id"]).reset_index(drop=True)

In [4]:
songs_data[["name", "artist"]].sample(5, random_state=1)

,name,artist
10971,what you were,the drums
10062,hand me down,matchbox twenty
6181,digital man,rush
9818,anything but you,way out west
13589,falling in fall,those dancing days


In [10]:
SONG = "hips don't lie"
ARTIST = "shakira"

recommender = HybridRecommenderSystem(number_of_recommendations=10, weight_content_based=0.5)

recommendations = recommender.give_recommendations(
    song_name=SONG,
    artist_name=ARTIST,
    songs_data=songs_data,
    track_ids=track_ids,
    transformed_matrix=transformed_matrix,
    interaction_matrix=interaction_matrix,
)
recommendations[["name", "artist", "spotify_preview_url"]]

,name,artist,spotify_preview_url
0,hips don't lie,shakira,https://p.scdn.co/mp3-preview/3859547944f57cfb...
1,alejandro,lady gaga,https://p.scdn.co/mp3-preview/fc79c08bc9f46cca...
2,party in the u.s.a.,the barden bellas,https://p.scdn.co/mp3-preview/f8c5ff9e6aebca46...
3,imma be,black eyed peas,https://p.scdn.co/mp3-preview/bb97ba1db66b37e2...
4,sexy bitch,david guetta,https://p.scdn.co/mp3-preview/282100e2282a5d4e...
5,halo,depeche mode,https://p.scdn.co/mp3-preview/78eac59c0b2a7067...
6,bring me to life,katherine jenkins,https://p.scdn.co/mp3-preview/8f57eebe270df712...
7,hey there delilah,plain white t's,https://p.scdn.co/mp3-preview/fbac064e4b0b9692...
8,already gone,kelly clarkson,https://p.scdn.co/mp3-preview/660bd02563cebe61...
9,heartbreak warfare,john mayer,https://p.scdn.co/mp3-preview/fec54783c7f773f2...


In [11]:
for weight in [0.9, 0.5, 0.1]:
    recs = HybridRecommenderSystem(5, weight).give_recommendations(
        SONG, ARTIST, songs_data, track_ids, transformed_matrix, interaction_matrix
    )
    print(f"\ncontent weight = {weight}")
    print(recs[["name", "artist"]].to_string(index=False))


content weight = 0.9
                   name      artist
         hips don't lie     shakira
              alejandro   lady gaga
             la tortura     shakira
       que me quedes tu     shakira
underneath your clothes     shakira
      hoedown throwdown miley cyrus

content weight = 0.5
               name            artist
     hips don't lie           shakira
          alejandro         lady gaga
party in the u.s.a. the barden bellas
            imma be   black eyed peas
         sexy bitch      david guetta
               halo      depeche mode

content weight = 0.1
               name            artist
     hips don't lie           shakira
          alejandro         lady gaga
party in the u.s.a. the barden bellas
               halo      depeche mode
            imma be   black eyed peas
         sexy bitch      david guetta
